<!-- cellID=intro -->
# DSL Draw: TTree::Draw Equivalent for RDataFrame

**RDataFrameDSL Tutorial - Phase 13.6.G+**

## Purpose

This notebook demonstrates `dsl.draw()` as a **TTree::Draw equivalent** with **enhanced capabilities** for RDataFrame.

## Why This Matters

ROOT's `TTree::Draw()` is being deprecated in favor of RDataFrame. However, RDataFrame lacks a direct `Draw()` equivalent for quick exploratory analysis. This DSL fills that gap.

## What This Notebook Covers

| Part | Topic | TTree::Draw | DSL | Notes |
|------|-------|-------------|-----|-------|
| **1** | Simple 1D histogram | ✅ `tree->Draw("x")` | ✅ `dsl.draw("x")` | Equivalent |
| **1** | 2D histogram (y:x) | ✅ `tree->Draw("y:x")` | ✅ `dsl.draw("y:x")` | Equivalent |
| **1** | Math expressions | ✅ `tree->Draw("x*2")` | ✅ `dsl.draw("x*2")` | Equivalent |
| **1** | Selection cuts | ✅ `tree->Draw("x","cut")` | ✅ `dsl.draw("x",selection)` | Equivalent |
| **2** | `RVec<RVec<T>>` nested | Wokgs | ✅ Works | **DSL advantage** |
| **2** | 2D cluster positions | ❌ Crashes | ✅ Works | **DSL advantage** |
| **3** | Slicing `[:3]` | ❌ Impossible | ✅ Works | **DSL-only** |
| **3** | Negative index `[-1]` | ❌ Impossible | ✅ Works | **DSL-only** |
| **3** | Step slice `[::2]` | ❌ Impossible | ✅ Works | **DSL-only** |
| **4** | Batch operations | ❌ Manual loop | ✅ `draw_batch()` | **DSL-only** |

## Key Benefits

1. **Familiar syntax**: `draw("y:x")` just like TTree::Draw
2. **Works with nested arrays**: `RVec<RVec<double>>` fully supported
3. **Extended capabilities**: Slicing, broadcast, reductions
4. **Batch operations**: Single extraction, multiple plots

<!-- cellID=setup_header -->
---
## Setup

In [ ]:
# cellID=setup_code
import numpy as np
import ROOT
import sys
import shutil
import time
sys.path.insert(0, '..')

from RDataFrameDSL import DSLCompiler
from tests.generators.toy_nd import generate_nd_2d_root, generate_nd_3d_root

# Set up plotting
import dfextensions.dfdraw as dfdraw

print("✅ Setup complete")

## Set draw style

In [ ]:
dfdraw.set_style({'figure.figsize': (5, 3),'stats.show': True})

<!-- cellID=generate_data_header -->
### Generate Test Data

Using **demo mode** for realistic physics distributions:
- `track_pt`: Exponential (mean 0.4 GeV)
- `track_eta`: Flat [-1, 1]
- `cluster_Q`: Landau-like energy loss (1/β²)

**5000 events** for measurable timing.

In [ ]:
# cellID=generate_data_code
# Generate 2D test data with realistic physics distributions
# Using 5000 events for measurable timing
nd_2d_file_tmp = generate_nd_2d_root(size="L", seed=42, n_events=5000, mode='demo')
nd_2d_file = 'nd_2d_demo.root'
shutil.copy(nd_2d_file_tmp, nd_2d_file)

# Create RDataFrame and TTree access
nd_2d_rdf = ROOT.RDataFrame("Events", nd_2d_file)
tfile2d = ROOT.TFile(nd_2d_file)
tree2d = tfile2d.Get("Events")

# Schema for DSL
nd_2d_schema = {
    'event_id': 'long',
    'n_tracks': 'int',
    'event_weight': 'double',
    'track_pt': 'RVec<double>',
    'track_eta': 'RVec<double>',
    'cluster_Q': 'RVec<RVec<double>>',
    'cluster_x': 'RVec<RVec<double>>',
    'cluster_y': 'RVec<RVec<double>>',
}

# Create TCanvas for ROOT comparisons
c2d = ROOT.TCanvas("c2d", "TTree::Draw", 500, 300)

# Create DSL compiler
dsl = DSLCompiler(nd_2d_schema,redefinition='ifneeded')

n_events = nd_2d_rdf.Count().GetValue()
print(f"✅ Generated {n_events} events with demo physics distributions")
print(f"   File: {nd_2d_file}")
print(f"   Columns: {list(nd_2d_schema.keys())}")

<!-- cellID=part1_header -->
---
# Part 1: TTree::Draw Equivalence

Each cell shows **both** TTree::Draw and DSL side-by-side with:
- Timing comparison
- Statistics comparison
- Equivalent options

<!-- cellID=1.1_header -->
### 1.1 Simple 1D Histogram

In [ ]:
# cellID=1.1_code
# === TTree::Draw ===
t0 = time.time()
entries = tree2d.Draw("track_pt>>h1(50,0,3)")
ttree_time = time.time() - t0
c2d.Draw()
h1 = tree2d.GetHistogram()

# === DSL equivalent ===
t0 = time.time()
fig, ax, stats = dsl.draw("track_pt", nd_2d_rdf, bins=50, range=(0, 3), title="Track pT (GeV)")
dsl_time = time.time() - t0

# === Comparison ===
print("="*60)
print("1.1 Simple 1D Histogram: track_pt")
print("="*60)
print(f"TTree::Draw: tree->Draw(\"track_pt>>h1(50,0,3)\")")
print(f"DSL:         dsl.draw('track_pt', rdf, bins=50, range=(0,3))")
print()
print(f"{'Metric':<12} {'TTree::Draw':>15} {'DSL':>15} {'Match':>10}")
print("-"*55)
print(f"{'Entries':<12} {int(h1.GetEntries()):>15} {stats['n']:>15} {'✅' if int(h1.GetEntries()) == stats['n'] else '❌':>10}")
print(f"{'Mean':<12} {h1.GetMean():>15.4f} {stats['mean']:>15.4f} {'✅' if abs(h1.GetMean() - stats['mean']) < 0.01 else '❌':>10}")
print(f"{'Std Dev':<12} {h1.GetStdDev():>15.4f} {stats['std']:>15.4f} {'✅' if abs(h1.GetStdDev() - stats['std']) < 0.01 else '❌':>10}")
print(f"{'Time (ms)':<12} {ttree_time*1000:>15.1f} {dsl_time*1000:>15.1f} {f'{dsl_time/ttree_time:.1f}x':>10}")

<!-- cellID=1.2_header -->
### 1.2 2D Histogram (y:x syntax)

In [ ]:
# cellID=1.2_code
# === TTree::Draw ===
t0 = time.time()
entries = tree2d.Draw("track_eta:track_pt>>h2(50,0,3,50,-1.5,1.5)", "", "colz")
ttree_time = time.time() - t0
c2d.Draw()
h2 = ROOT.gDirectory.Get("h2")

# === DSL equivalent ===
t0 = time.time()
fig, ax, stats = dsl.draw("track_eta:track_pt", nd_2d_rdf, 
                          type='hist2d', bins=50, 
                          range=[[0, 3], [-1.5, 1.5]],
                          title="η vs pT")
dsl_time = time.time() - t0

# === Comparison ===
print("="*60)
print("1.2 2D Histogram: track_eta:track_pt")
print("="*60)
print(f"TTree::Draw: tree->Draw(\"track_eta:track_pt>>h2(...)\", \"\", \"colz\")")
print(f"DSL:         dsl.draw('track_eta:track_pt', rdf, type='hist2d')")
print()
print(f"{'Metric':<12} {'TTree::Draw':>15} {'DSL':>15} {'Match':>10}")
print("-"*55)
print(f"{'Entries':<12} {int(h2.GetEntries()):>15} {stats['n']:>15} {'✅' if int(h2.GetEntries()) == stats['n'] else '❌':>10}")
print(f"{'Time (ms)':<12} {ttree_time*1000:>15.1f} {dsl_time*1000:>15.1f} {f'{dsl_time/ttree_time:.1f}x':>10}")

<!-- cellID=1.3_header -->
### 1.3 Math Expression

In [ ]:
# cellID=1.3_code
# === TTree::Draw ===
t0 = time.time()
entries = tree2d.Draw("track_pt*track_pt>>h3(50,0,9)")
ttree_time = time.time() - t0
c2d.Draw()
h3 = tree2d.GetHistogram()

# === DSL equivalent ===
dsl.define("pt2", "track_pt*track_pt")
t0 = time.time()
fig, ax, stats = dsl.draw("pt2", nd_2d_rdf, bins=50, range=(0, 9), title="Track pT²")
dsl_time = time.time() - t0

# === Comparison ===
print("="*60)
print("1.3 Math Expression: track_pt²")
print("="*60)
print(f"TTree::Draw: tree->Draw(\"track_pt*track_pt\")")
print(f"DSL:         dsl.define('pt2', 'track_pt*track_pt'); dsl.draw('pt2')")
print()
print(f"{'Metric':<12} {'TTree::Draw':>15} {'DSL':>15} {'Match':>10}")
print("-"*55)
print(f"{'Entries':<12} {int(h3.GetEntries()):>15} {stats['n']:>15} {'✅' if int(h3.GetEntries()) == stats['n'] else '❌':>10}")
print(f"{'Mean':<12} {h3.GetMean():>15.4f} {stats['mean']:>15.4f} {'✅' if abs(h3.GetMean() - stats['mean']) < 0.01 else '❌':>10}")
print(f"{'Time (ms)':<12} {ttree_time*1000:>15.1f} {dsl_time*1000:>15.1f} {f'{dsl_time/ttree_time:.1f}x':>10}")

<!-- cellID=part2_header -->
---
# Part 2: Where TTree::Draw FAILS - Nested Arrays

**This is the key limitation.** TTree::Draw cannot handle `RVec<RVec<T>>` properly.

| Column | Type | TTree::Draw | DSL |
|--------|------|-------------|-----|
| `track_pt` | `RVec<double>` | ✅ Works | ✅ Works |
| `cluster_Q` | `RVec<RVec<double>>` | Works | ✅ Works |

In [ ]:
# cellID=2.1_code
# === 2.1 Nested Array: cluster_Q (RVec<RVec<double>>) ===
c=ROOT.TCanvas("","",500,300)
print("="*60)
print("2.1 Nested Array: cluster_Q (RVec<RVec<double>>)")
print("="*60)
print()

# === TTree::Draw (2 calls) ===
t0 = time.time()
tree2d.Draw("cluster_Q>>hq(50,0,500)")
ttree_time1 = time.time() - t0
t0 = time.time()
tree2d.Draw("cluster_Q>>hq2(50,0,500)")
ttree_time2 = time.time() - t0
c.Draw()
hq = tree2d.GetHistogram()

# === RDataFrame AsNumpy (2 calls) ===
t0 = time.time()
rdf_data = nd_2d_rdf.AsNumpy(["cluster_Q"])
rdf_time1 = time.time() - t0
t0 = time.time()
rdf_data = nd_2d_rdf.AsNumpy(["cluster_Q"])
rdf_time2 = time.time() - t0

# === DSL to_pandas (2 calls) ===
t0 = time.time()
df = dsl.to_pandas(nd_2d_rdf, columns=["cluster_Q"])
topandas_time1 = time.time() - t0
t0 = time.time()
df = dsl.to_pandas(nd_2d_rdf, columns=["cluster_Q"])
topandas_time2 = time.time() - t0

# === DSL draw (2 calls) ===
import matplotlib.pyplot as plt
t0 = time.time()
fig, ax, stats = dsl.draw("cluster_Q", nd_2d_rdf, bins=50, range=(0, 500), title="Cluster Charge Q")
dsl_time1 = time.time() - t0
plt.close(fig)
t0 = time.time()
fig, ax, stats = dsl.draw("cluster_Q", nd_2d_rdf, bins=50, range=(0, 500), title="Cluster Charge Q")
dsl_time2 = time.time() - t0

# === Comparison ===
print(f"TTree::Draw: tree->Draw(\"cluster_Q>>hq(50,0,500)\")")
print(f"DSL:         dsl.draw('cluster_Q', rdf, bins=50, range=(0,500))")
print()
print(f"{'Metric':<15} {'TTree::Draw':>12} {'RDF AsNumpy':>12} {'to_pandas':>12} {'dsl.draw':>12}")
print("-"*70)
print(f"{'Entries':<15} {int(hq.GetEntries()):>12} {'-':>12} {len(df):>12} {stats['n']:>12}")
print(f"{'Mean':<15} {hq.GetMean():>12.2f} {'-':>12} {df['cluster_Q'].mean():>12.2f} {stats['mean']:>12.2f}")
print(f"{'Time 1 (ms)':<15} {ttree_time1*1000:>12.1f} {rdf_time1*1000:>12.1f} {topandas_time1*1000:>12.1f} {dsl_time1*1000:>12.1f}")
print(f"{'Time 2 (ms)':<15} {ttree_time2*1000:>12.1f} {rdf_time2*1000:>12.1f} {topandas_time2*1000:>12.1f} {dsl_time2*1000:>12.1f}")
print()
print("⚠️  Mean differs: TTree::Draw computes stats within range, DSL uses full data")
print("    TODO: Fix DSL to compute stats within specified range")

In [ ]:
# cellID=2.2_code
# === 2D scatter with nested arrays ===
tree=tree2d
print("="*60)
print("2.2 2D Scatter: cluster_y:cluster_x (both RVec<RVec<double>>)")
print("="*60)
print()

# TTree::Draw
tree.Draw("cluster_y:cluster_x>>hxy(600,-300,300,100,-300,300)", "", "goff")
hxy = ROOT.gDirectory.Get("hxy")
t0 = time.time()
tree.Draw("cluster_y:cluster_x>>hxy2(600,-300,300,100,-300,300)", "", "goff")
ttree_time1 = time.time() - t0
t0 = time.time()
c.cd()
tree.Draw("cluster_y:cluster_x>>hxy3(600,-300,300,100,-300,300)", "", "")
ttree_time2 = time.time() - t0
c.Draw()
ttree_entries = int(hxy.GetEntries())
ttree_mean_x = hxy.GetMean(1)
ttree_mean_y = hxy.GetMean(2)

print(f"TTree::Draw: tree->Draw('cluster_y:cluster_x>>hxy(100,-300,300,100,-300,300)')")

# DSL
t0 = time.time()
fig, ax, stats = dsl.draw("cluster_y:cluster_x", nd_2d_rdf,
                          type='hist2d', bins=300, range=[(-300,300),(-300,300)],
                          title="Cluster Positions (x vs y)")
dsl_time1 = time.time() - t0
t0 = time.time()
fig2, ax2, stats2 = dsl.draw("cluster_y:cluster_x", nd_2d_rdf,
                             type='hist2d', bins=300, range=[(-300,300),(-300,300)],
                             title="Cluster Positions (x vs y)")
dsl_time2 = time.time() - t0
plt.close(fig2)
c.Draw()
print(f"DSL:         dsl.draw('cluster_y:cluster_x', rdf, type='hist2d', bins=100, range=[(-300,300),(-300,300)])")
print()

# Comparison table
print(f"{'Metric':<20} {'TTree::Draw':>15} {'DSL':>15} {'Match':>10}")
print("-"*60)
print(f"{'Entries':<20} {ttree_entries:>15} {stats['n']:>15} {'✅' if ttree_entries == stats['n'] else '❌':>10}")
print(f"{'Mean X':<20} {ttree_mean_x:>15.2f} {stats.get('mean_x', 0):>15.2f} {'✅' if abs(ttree_mean_x - stats.get('mean_x', 0)) < 1 else '⚠️':>10}")
print(f"{'Mean Y':<20} {ttree_mean_y:>15.2f} {stats.get('mean_y', 0):>15.2f} {'✅' if abs(ttree_mean_y - stats.get('mean_y', 0)) < 1 else '⚠️':>10}")
print(f"{'Time 1 (ms)':<20} {ttree_time1*1000:>15.1f} {dsl_time1*1000:>15.1f} {f'{dsl_time1/ttree_time1:.1f}x':>10}")
print(f"{'Time 2 (ms)':<20} {ttree_time2*1000:>15.1f} {dsl_time2*1000:>15.1f} {f'{dsl_time2/ttree_time2:.1f}x':>10}")

<!-- cellID=part3_header -->
---
# Part 3: DSL-Only Features - Slicing

These operations are **impossible** with TTree::Draw.

In [ ]:
# cellID=3.1_code
# === 3.1 First N elements: track_pt[:3] ===
print("="*60)
print("3.1 Slicing: First 3 tracks per event - track_pt[:3]")
print("="*60)
print()
print("TTree::Draw: tree->Draw(\"track_pt[:3]\")")
print("  ❌ IMPOSSIBLE - TTree::Draw does not support slicing")
print()

# DSL works
dsl.define("first_3_pt", "track_pt[:3]")
t0 = time.time()
fig, ax, stats = dsl.draw("first_3_pt", nd_2d_rdf, bins=50, range=(0, 3), title="First 3 tracks pT")
dsl_time = time.time() - t0

print(f"DSL: dsl.define('first_3_pt', 'track_pt[:3]')")
print(f"  ✅ Entries: {stats['n']} (max 3 per event)")
print(f"  Time: {dsl_time*1000:.1f} ms")

In [ ]:
# cellID=3.2_code
# === 3.2 Last element: track_pt[-1] ===
print("="*60)
print("3.2 Negative Index: Last track per event - track_pt[-1]")
print("="*60)
print()
print("TTree::Draw: ❌ IMPOSSIBLE")
print()

# DSL works
dsl.define("last_pt", "track_pt[-1]")
t0 = time.time()
fig, ax, stats = dsl.draw("last_pt", nd_2d_rdf, bins=50, range=(0, 3), title="Last track pT per event")
dsl_time = time.time() - t0

print(f"DSL: dsl.define('last_pt', 'track_pt[-1]')")
print(f"  ✅ Entries: {stats['n']} (1 per event)")
print(f"  Time: {dsl_time*1000:.1f} ms")

In [ ]:
# cellID=3.3_code
# === 3.3 Step slicing: track_pt[::2] ===
print("="*60)
print("3.3 Step Slicing: Every other track - track_pt[::2]")
print("="*60)
print()
print("TTree::Draw: ❌ IMPOSSIBLE")
print()

# DSL works
dsl.define("even_tracks", "track_pt[::2]")
t0 = time.time()
fig, ax, stats = dsl.draw("even_tracks", nd_2d_rdf, bins=50, range=(0, 3), title="Every other track pT")
dsl_time = time.time() - t0

print(f"DSL: dsl.define('even_tracks', 'track_pt[::2]')")
print(f"  ✅ Entries: {stats['n']} (~half of all tracks)")
print(f"  Time: {dsl_time*1000:.1f} ms")

<!-- cellID=part4_header -->
---
# Part 4: Batch Operations

`draw_batch()` creates multiple plots with a **single data extraction**.

In [ ]:
# cellID=4.1_code
print("="*60)
print("4.1 Batch Operations: Multiple plots, single extraction")
print("="*60)
print()

# Define multiple plots
specs = {
    'pt_dist': {'expr': 'track_pt', 'bins': 50, 'range': (0, 3), 'title': 'Track pT'},
    'eta_dist': {'expr': 'track_eta', 'bins': 50, 'range': (-1.5, 1.5), 'title': 'Track η'},
}

# === TTree::Draw equivalent (manual loop) ===
t0 = time.time()
tree2d.Draw("track_pt>>hbatch1(50,0,3)", "", "goff")
tree2d.Draw("track_eta>>hbatch2(50,-1.5,1.5)", "", "goff")
ttree_time = time.time() - t0

# === DSL batch ===
t0 = time.time()
results = dsl.draw_batch(specs, nd_2d_rdf)
dsl_time = time.time() - t0

print(f"TTree::Draw: 2 separate Draw() calls")
print(f"  Time: {ttree_time*1000:.1f} ms")
print()
print(f"DSL: dsl.draw_batch(specs, rdf)")
print(f"  Time: {dsl_time*1000:.1f} ms")
print(f"  Plots: {len(results)}")
for name, result in results.items():
    print(f"    {name}: {result['stats']['n']} entries")

<!-- cellID=summary -->
---
# Summary

## Feature Comparison

| Feature | TTree::Draw | RDataFrame | DSL |
|---------|-------------|------------|-----|
| Simple 1D histogram | ✅ | Manual | ✅ |
| 2D histogram (y:x) | ✅ | Manual | ✅ |
| Math expressions | ✅ | ✅ Define | ✅ |
| `RVec<double>` | ✅ | ✅ | ✅ |
| `RVec<RVec<double>>` | ❌ **Broken** | Manual | ✅ |
| Array slicing `[:3]` | ❌ | Manual | ✅ |
| Negative index `[-1]` | ❌ | Manual | ✅ |
| Step slice `[::2]` | ❌ | Manual | ✅ |
| Batch operations | Manual loop | Manual | ✅ |
| Statistics output | `GetMean()` etc | Manual | ✅ Auto |
| Python integration | ❌ | ✅ | ✅ |

## Key Takeaways

1. **DSL is a drop-in replacement** for TTree::Draw with familiar syntax
2. **DSL handles nested arrays** (`RVec<RVec<T>>`) that break TTree::Draw
3. **DSL adds slicing capabilities** impossible in TTree::Draw
4. **DSL provides batch operations** for efficient multi-plot workflows
5. **Statistics match exactly** - DSL is numerically equivalent

## For ROOT Team

This demonstrates a gap in RDataFrame's Python API. Consider:
- Adding a `Draw()` method to RDataFrame Python bindings
- Supporting nested `RVec<RVec<T>>` flattening in `AsNumpy()`
- Adding slice syntax to the RDataFrame DSL

In [ ]:
# cellID=cleanup
# Cleanup
tfile2d.Close()
print("✅ Notebook complete")

In [ ]:
# cellID=backup_start
#Backup

In [ ]:
# cellID=backup_28
# =============================================================================
# BACKUP/DEBUG: Performance Profiling
# =============================================================================
# This section profiles to_pandas() to identify flattening bottleneck
# Results saved to file for tracking optimization progress

import cProfile
import pstats
from io import StringIO
from datetime import datetime

print("="*60)
print("Performance Profiling: to_pandas() with RVec<RVec<double>>")
print("="*60)
print()

# === Profile to_pandas ===
pr = cProfile.Profile()
pr.enable()
df = dsl.to_pandas(nd_2d_rdf, columns=["cluster_Q"])
pr.disable()

# === Save to file ===
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
profile_file = f"profile_to_pandas_{timestamp}.prof"
pr.dump_stats(profile_file)
print(f"✅ Profile saved to: {profile_file}")
print(f"   Load with: pstats.Stats('{profile_file}').sort_stats('cumulative').print_stats(20)")
print()

# === Print top 20 time consumers ===
print("Top 20 time consumers (cumulative):")
print("-"*60)
s = StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumulative')
ps.print_stats(20)
print(s.getvalue())

# === Print top 20 by total time (self, excluding subcalls) ===
print("\nTop 20 by self time (excluding subcalls):")
print("-"*60)
s2 = StringIO()
ps2 = pstats.Stats(pr, stream=s2).sort_stats('tottime')
ps2.print_stats(20)
print(s2.getvalue())

# === Summary ===
print("="*60)
print("Summary")
print("="*60)
print(f"Total rows flattened: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()
print("To view profile interactively:")
print(f"  python -c \"import pstats; pstats.Stats('{profile_file}').sort_stats('cumulative').print_stats(30)\"")
print()
print("To visualize with snakeviz:")
print(f"  pip install snakeviz && snakeviz {profile_file}")

In [ ]:
# cellID=backup_29
from RDataFrameDSL.flatten import awkward_available, _flatten_numpy_2level
print(f"Awkward available: {awkward_available()}")

# Add timing inside the function
import time
rdf=nd_2d_rdf
data = rdf.AsNumpy(['event_id', 'cluster_Q'])
t0 = time.time()
from RDataFrameDSL.flatten import flatten_to_dataframe
df = flatten_to_dataframe(data, columns=['cluster_Q'])
print(f"flatten_to_dataframe: {(time.time()-t0)*1000:.1f} ms")
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")

In [ ]:
# cellID=backup_30
from RDataFrameDSL.flatten import (
    _flatten_numpy_2level_awkward, 
    _flatten_numpy_2level_loop,
    awkward_available
)
import time
import numpy as np

data = rdf.AsNumpy(['event_id', 'cluster_Q'])

# Test Awkward path directly
t0 = time.time()
result_awk = _flatten_numpy_2level_awkward(
    data, ['cluster_Q'], 'event_id', None
)
t_awk = (time.time() - t0) * 1000

# Test loop path directly  
t0 = time.time()
result_loop = _flatten_numpy_2level_loop(
    data, ['cluster_Q'], 'event_id', None
)
t_loop = (time.time() - t0) * 1000

print(f"Awkward path: {t_awk:.1f} ms")
print(f"Loop path:    {t_loop:.1f} ms")
print(f"Speedup:      {t_loop/t_awk:.1f}x")

In [ ]:
# cellID=backup_31
tree2d.Draw("cluster_x")
c2d.Draw()

In [ ]:
# cellID=backup_32
print(dsl.describe_structure(0xFFFF))


In [ ]:
from dfextensions.dfdraw import DFDraw, set_style

set_style({'stats.show': True,    'stats.position': 'upper left',})

In [ ]:
from dfextensions.dfdraw import DFDraw
df = dsl.to_pandas(nd_2d_rdf, columns=['track_pt'])
plotter = DFDraw(df)
fig, ax, stats = plotter.hist('track_pt', bins=50, range=(0, 3))
#plotter.add_statistics_box(ax, df['track_pt'].values)



In [ ]:
plotter.add_statistics_box(ax,stats)

In [ ]:
stats